In [3]:
from lexguard.utils.paths import RAW_DATA_DIR
from lexguard.ingestion.pdf_loader import load_pdf
from lexguard.ingestion.text_loader import load_text
from lexguard.ingestion.chunker import build_chunks

USE_PDF = False  

if USE_PDF:
    file_path = RAW_DATA_DIR / "sample.pdf"
    loader = load_pdf
else:
    file_path = RAW_DATA_DIR / "sample_policy.txt"
    loader = load_text

print("Using file:", file_path)
print("Exists:", file_path.exists())

if not file_path.exists():
    raise FileNotFoundError(f"File not found: {file_path}")

pages = loader(str(file_path))

chunks = build_chunks(
    document_id="doc_1",
    title="Sample Policy",
    pages=pages
)

print("\nTotal chunks:", len(chunks))

for c in chunks:
    print("-" * 50)
    print("clause_id:", c.clause_id)
    print("section_id:", c.section_id)
    print("section_title:", c.section_title)
    print("text:", c.chunk_text)

Using file: /workspaces/lexguard/data/raw/sample_policy.txt
Exists: True

Total chunks: 3
--------------------------------------------------
clause_id: 1_1
section_id: 1
section_title: Data Retention
text: The contractor shall retain all audit logs for a minimum period of 90 days.
--------------------------------------------------
clause_id: 1_3
section_id: 2
section_title: Exceptions
text: Unless otherwise approved in writing by the compliance officer, logs related to financial transactions must be retained for 180 days.
--------------------------------------------------
clause_id: 1_5
section_id: 3
section_title: Deletion
text: All temporary logs may be deleted after 30 days.


In [4]:
import json
from pathlib import Path

from lexguard.utils.paths import PROJECT_ROOT
from lexguard.retrieval.bm25 import BM25Retriever
from lexguard.retrieval.dense import DenseRetriever
from lexguard.retrieval.hybrid import HybridRetriever

benchmark_path = PROJECT_ROOT / "data" / "benchmarks" / "retrieval_smoke_test.json"

with open(benchmark_path, "r", encoding="utf-8") as f:
    benchmark = json.load(f)

bm25 = BM25Retriever(chunks)
dense = DenseRetriever(chunks)
hybrid = HybridRetriever(chunks, bm25_weight=0.5, dense_weight=0.5)

systems = {
    "bm25": bm25,
    "dense": dense,
    "hybrid": hybrid,
}

results = []

for item in benchmark:
    query = item["query"]
    expected = item["expected_section"]

    row = {
        "query": query,
        "expected": expected,
    }

    for name, system in systems.items():
        pred = system.query(query, top_k=1)[0].section_title
        row[name] = pred
        row[f"{name}_hit"] = int(pred == expected)

    results.append(row)

for row in results:
    print("=" * 80)
    print("QUERY:", row["query"])
    print("EXPECTED:", row["expected"])
    print("BM25   :", row["bm25"], "| hit =", row["bm25_hit"])
    print("DENSE  :", row["dense"], "| hit =", row["dense_hit"])
    print("HYBRID :", row["hybrid"], "| hit =", row["hybrid_hit"])

print("\nSummary")
for name in systems:
    score = sum(r[f"{name}_hit"] for r in results) / len(results)
    print(f"{name}: hit@1 = {score:.2f}")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4020.15it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4410.50it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


QUERY: log retention period
EXPECTED: Data Retention
BM25   : Data Retention | hit = 1
DENSE  : Exceptions | hit = 0
HYBRID : Data Retention | hit = 1
QUERY: financial logs retention
EXPECTED: Exceptions
BM25   : Exceptions | hit = 1
DENSE  : Exceptions | hit = 1
HYBRID : Exceptions | hit = 1
QUERY: log deletion policy
EXPECTED: Deletion
BM25   : Data Retention | hit = 0
DENSE  : Exceptions | hit = 0
HYBRID : Data Retention | hit = 0
QUERY: how long should logs be stored
EXPECTED: Data Retention
BM25   : Deletion | hit = 0
DENSE  : Exceptions | hit = 0
HYBRID : Deletion | hit = 0

Summary
bm25: hit@1 = 0.50
dense: hit@1 = 0.25
hybrid: hit@1 = 0.50
